# **Build a Mini-BERT Model From Scratch**

In [28]:
#Import Libraries

import torch
import torch.nn as nn 
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import IMDB
from transformers import BertTokenizer
import requests
from zipfile import ZipFile
from datasets import Dataset
import pandas as pd
import json
from torchtext.functional import pad_sequence

In [3]:
train_iter, val_iter = IMDB()


In [5]:
#define tokenizor
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [9]:
def download (url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            f.write(response.content)
    else:
        print(f"Failed to download. Status code: {response.status_code}")

In [11]:
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/bZaoQD52DcMpE7-kxwAG8A.zip'
filename = 'BERT-dataset'

In [12]:
download(url, filename)

In [15]:
def unzip_file (filename,directory):
    with ZipFile(filename, 'r') as unzip_f:
        unzip_f.extractall(directory)

In [18]:
directory = 'D:\Projects\mini-bert-nsp-mlm-pretraining'

In [19]:
unzip_file(filename,directory)

In [24]:
data = pd.read_csv('bert_dataset/bert_train_data.csv')
data.iloc[2]

Original Text    I rented I AM CURIOUS-YELLOW from my video sto...
BERT Input       [1, 75, 7, 5, 405, 8, 1003, 148, 34, 178, 3, 2...
BERT Label       [0, 0, 0, 0, 405, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0...
Segment Label    1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,...
Is Next                                                          1
Name: 2, dtype: object

### **Build Dataset Class**

In [ ]:
class BertDatatsetCSV(Dataset):
    def __init__(self,filename):
        self.data = pd.read_csv(filename) 
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
     
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self,idx):
        row =  self.data.iloc[idx]
        
        try:
            
            bert_input = torch.tensor(json.loads(row['BERT input']), dtype=torch.long)
            bert_label = torch.tensor(json.loads(row['BERT label']), dtype=torch.long)
            segment_label = torch.tensor([int(x) for x in row['Segment Label'].split(',')], dtype=torch.long) 
            is_Next = torch.tensor(row['is Next'], dtype=torch.long)
            original_text = row['Original Text']
            
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON for row {idx}: {e}")
            print("BERT Input:", row['BERT Input'])
            print("BERT Label:", row['BERT Label'])
            # Handle the error, e.g., by skipping this row or using default values
            return None  # or some default values
    
        #Tokenize the original text
        #This creates:
            #input_ids → token numbers
            #attention_mask → tells model what is padding
        encoded_text = self.tokenizer.encode_plus(
            original_text,
            add_special_tokens = True,
            max_length = 512,
            padding = "max_length",
            truncation = True,
            return_tensor = 'pt'
        )
        
        #Why squeeze()?
            #return_tensors="pt" adds batch dimension → shape (1, 512)
            #.squeeze() → (512,)

        input_ids = encoded_text['input_ids'].squeeze()
        attention_mask = encoded_text['attention_mask'].squeeze()
    
        return (bert_input, bert_label, segment_label, is_Next,input_ids,attention_mask,original_text)
         

### **Build Collate Function**

In [29]:
PAD_IDX = 0

def collate_func(batch):
    bert_input_batch, bert_label_batch,segement_label_batch, is_next_batch, input_ids_batch, attention_mask_batch, original_text_batch = [],[],[],[],[],[],[]
    
    for bert_input, bert_label,segment_label, is_next, input_ids, attention_mask, original_text in batch:
        
        bert_input_batch.append(torch.tensor(bert_input, dtype=torch.long))
        bert_label_batch.append(torch.tensor(bert_label,dtype=torch.long))
        segement_label_batch.append(torch.tensor(segment_label,dtype=torch.long))
        is_next_batch.append(is_next)
        input_ids_batch.append(input_ids)
        attention_mask_batch.append(attention_mask)
        original_text_batch.append(original_text)
        
    bert_input_batch = pad_sequence(bert_input_batch,batch_first=True,padding_value=PAD_IDX)
    bert_label_batch = pad_sequence(bert_label_batch,batch_first=True,padding_value=PAD_IDX)
    segement_label_batch = pad_sequence(segement_label_batch,batch_first=True,padding_value=PAD_IDX)
    is_next_batch = torch.tensor(is_next_batch,dtype=torch.long)
    
    return bert_input_batch,bert_label_batch,segement_label_batch,is_next_batch